In [2]:
!pip install chronos-forecasting
!pip install -U chronos-forecasting "timesfm[xreg]" openpyxl scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/25.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/25.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.0 MB ? eta -:--:--
    --------------------------------------- 0.5/25.0 MB 730.2 kB/s eta 0:00:34
    --------------------------------------- 0.5/25.0 MB 730.2 kB/s eta 0:00:34
   - -------------------------------------- 0.8/25.0 MB 621.9 kB/s eta 0:00:39
   - -------------------------------------- 0.8/25.0 MB 621.9 kB/s eta 0:00:39
   - -------------------------------------- 1.0/25.0 MB 621.7 kB/s eta 0:00:39
   - -------------------------------------- 1.0/25.0 MB 621.7 kB/s eta 0:00:39
   -- ------------------------------------- 1.3/25.0 MB 627.5 kB/s eta 0:00:38
   -- ------------------------------------- 1.3/25.0 MB 627.5 kB/s eta 0:00:38
   -- ------------------------------------- 1.6/25.0 MB 621.5 kB/s eta 0:00:38
   -- ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.8 requires click==8.1.8, but you have click 8.3.3 which is incompatible.
litellm 1.83.8 requires importlib-metadata==8.5.0, but you have importlib-metadata 6.11.0 which is incompatible.
litellm 1.83.8 requires openai==2.24.0, but you have openai 1.109.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =========================================================
# FULL SIMPLE TIME SERIES PROJECT
# Target: order_qty
# Models: Chronos-2 or TimesFM 2.5
# Data: Daily_Orders / Hourly_Orders from your Excel file
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 0) SETTINGS
# ---------------------------------------------------------
FILE_PATH = "complex_multi_product_order_forecasting_dataset.xlsx"
RUN_MODEL = "chronos"   # change to "timesfm" if you want TimesFM

DAILY_HORIZON = 30
HOURLY_HORIZON = 48

TARGET_COL = "order_qty"

# ---------------------------------------------------------
# 1) LOAD DATA
# ---------------------------------------------------------
daily_df = pd.read_excel(FILE_PATH, sheet_name="Daily_Orders", header=2)
hourly_df = pd.read_excel(FILE_PATH, sheet_name="Hourly_Orders", header=2)

# clean column names
daily_df.columns = daily_df.columns.str.strip()
hourly_df.columns = hourly_df.columns.str.strip()

# timestamp to datetime
daily_df["timestamp"] = pd.to_datetime(daily_df["timestamp"])
hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])

# create series id
daily_df["series_id"] = daily_df["company_code"].astype(str) + "_" + daily_df["product_id"].astype(str)
hourly_df["series_id"] = hourly_df["company_code"].astype(str) + "_" + hourly_df["product_id"].astype(str)

# ---------------------------------------------------------
# 2) COVARIATES
# ---------------------------------------------------------
daily_covariates = [
    "price", "stock_available", "promotion", "holiday",
    "is_weekend", "day_of_week", "month", "temperature_c", "rainfall_mm"
]

hourly_covariates = [
    "price", "promotion", "holiday", "is_weekend",
    "day_of_week", "month", "hour", "business_hour",
    "evening_peak", "temperature_c", "rainfall_mm"
]

# ---------------------------------------------------------
# 3) KEEP ONLY REQUIRED COLUMNS
# ---------------------------------------------------------
daily_model_df = daily_df[["timestamp", "series_id", TARGET_COL] + daily_covariates].copy()
hourly_model_df = hourly_df[["timestamp", "series_id", TARGET_COL] + hourly_covariates].copy()

# sort
daily_model_df = daily_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)
hourly_model_df = hourly_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

# ---------------------------------------------------------
# 4) TRAIN-TEST SPLIT
# ---------------------------------------------------------
def split_by_series(df, horizon):
    train_parts = []
    test_parts = []

    for sid, g in df.groupby("series_id"):
        g = g.sort_values("timestamp").reset_index(drop=True)
        if len(g) <= horizon + 5:
            continue
        train_parts.append(g.iloc[:-horizon].copy())
        test_parts.append(g.iloc[-horizon:].copy())

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    return train_df, test_df

daily_train, daily_test = split_by_series(daily_model_df, DAILY_HORIZON)
hourly_train, hourly_test = split_by_series(hourly_model_df, HOURLY_HORIZON)

# ---------------------------------------------------------
# 5) METRICS
# ---------------------------------------------------------
def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def wmape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom * 100

def make_result_table(actual_df, pred_df, pred_col="predicted_order_qty"):
    return actual_df.merge(
        pred_df[["series_id", "timestamp", "predicted_order_qty"]],
        on=["series_id", "timestamp"],
        how="left"
    )

# ---------------------------------------------------------
# 6) CHRONOS MODEL
# ---------------------------------------------------------
def run_chronos(train_df, test_df, covariates, horizon):
    from chronos import Chronos2Pipeline

    pipeline = Chronos2Pipeline.from_pretrained(
        "amazon/chronos-2",
        device_map="cpu"
    )

    context_df = train_df[["timestamp", "series_id", TARGET_COL] + covariates].copy()
    future_df = test_df[["timestamp", "series_id"] + covariates].copy()
    actual_df = test_df[["timestamp", "series_id", TARGET_COL]].copy()

    pred_df = pipeline.predict_df(
        context_df=context_df,
        future_df=future_df,
        prediction_length=horizon,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="series_id",
        timestamp_column="timestamp",
        target=TARGET_COL,
    )

    # pick point forecast column
    if "predictions" in pred_df.columns:
        pred_col = "predictions"
    elif "0.5" in pred_df.columns:
        pred_col = "0.5"
    else:
        pred_col = pred_df.columns[-1]

    pred_df = pred_df.rename(columns={pred_col: "predicted_order_qty"})
    result_df = actual_df.merge(
        pred_df[["series_id", "timestamp", "predicted_order_qty"]],
        on=["series_id", "timestamp"],
        how="left"
    )

    return result_df

# ---------------------------------------------------------
# 7) TIMESFM MODEL
# ---------------------------------------------------------
def run_timesfm(train_df, test_df, num_covs, cat_covs, static_covs, horizon):
    import torch
    import timesfm

    torch.set_float32_matmul_precision("high")

    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
        "google/timesfm-2.5-200m-pytorch"
    )

    model.compile(
        timesfm.ForecastConfig(
            max_context=1024,
            max_horizon=256,
            normalize_inputs=True,
            use_continuous_quantile_head=True,
            fix_quantile_crossing=True,
        )
    )

    inputs = []
    dyn_num = {c: [] for c in num_covs}
    dyn_cat = {c: [] for c in cat_covs}
    static_cat = {c: [] for c in static_covs}
    actual_blocks = []

    for sid in sorted(train_df["series_id"].unique()):
        tr = train_df[train_df["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        te = test_df[test_df["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        full = pd.concat([tr, te], ignore_index=True)

        inputs.append(tr[TARGET_COL].astype(float).to_numpy())

        for c in num_covs:
            dyn_num[c].append(full[c].astype(float).to_numpy())
        for c in cat_covs:
            dyn_cat[c].append(full[c].astype(float).to_numpy())
        for c in static_covs:
            static_cat[c].append(str(tr[c].iloc[0]))

        actual_blocks.append(te[["timestamp", "series_id", TARGET_COL]].copy())

    actual_df = pd.concat(actual_blocks, ignore_index=True)

    point, quantiles = model.forecast_with_covariates(
        inputs=inputs,
        dynamic_numerical_covariates=dyn_num,
        dynamic_categorical_covariates=dyn_cat,
        static_categorical_covariates=static_cat,
        xreg_mode="xreg + timesfm"
    )

    rows = []
    series_ids = sorted(train_df["series_id"].unique())

    for i, sid in enumerate(series_ids):
        te = test_df[test_df["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        pred_vals = point[i][:len(te)]

        tmp = te[["timestamp", "series_id", TARGET_COL]].copy()
        tmp["predicted_order_qty"] = pred_vals
        rows.append(tmp)

    result_df = pd.concat(rows, ignore_index=True)
    return result_df

# ---------------------------------------------------------
# 8) RUN DAILY + HOURLY
# ---------------------------------------------------------
if RUN_MODEL.lower() == "chronos":
    daily_result = run_chronos(daily_train, daily_test, daily_covariates, DAILY_HORIZON)
    hourly_result = run_chronos(hourly_train, hourly_test, hourly_covariates, HOURLY_HORIZON)

elif RUN_MODEL.lower() == "timesfm":
    daily_num_covs = ["price", "stock_available", "temperature_c", "rainfall_mm"]
    daily_cat_covs = ["promotion", "holiday", "is_weekend", "day_of_week", "month"]
    daily_static_covs = ["company_code", "region", "category", "channel"]

    hourly_num_covs = ["price", "temperature_c", "rainfall_mm"]
    hourly_cat_covs = ["promotion", "holiday", "is_weekend", "day_of_week", "month", "hour", "business_hour", "evening_peak"]
    hourly_static_covs = ["company_code", "region", "category", "channel"]

    daily_result = run_timesfm(daily_train, daily_test, daily_num_covs, daily_cat_covs, daily_static_covs, DAILY_HORIZON)
    hourly_result = run_timesfm(hourly_train, hourly_test, hourly_num_covs, hourly_cat_covs, hourly_static_covs, HOURLY_HORIZON)

else:
    raise ValueError("RUN_MODEL must be 'chronos' or 'timesfm'")

# ---------------------------------------------------------
# 9) ADD ERROR COLUMNS
# ---------------------------------------------------------
daily_result["abs_error"] = np.abs(daily_result[TARGET_COL] - daily_result["predicted_order_qty"])
daily_result["ape"] = np.where(
    daily_result[TARGET_COL] != 0,
    daily_result["abs_error"] / np.abs(daily_result[TARGET_COL]),
    np.nan
)

hourly_result["abs_error"] = np.abs(hourly_result[TARGET_COL] - hourly_result["predicted_order_qty"])
hourly_result["ape"] = np.where(
    hourly_result[TARGET_COL] != 0,
    hourly_result["abs_error"] / np.abs(hourly_result[TARGET_COL]),
    np.nan
)

# ---------------------------------------------------------
# 10) FINAL METRICS
# ---------------------------------------------------------
print("\n====================")
print("DAILY METRICS")
print("====================")
print("MAPE :", round(mape(daily_result[TARGET_COL], daily_result["predicted_order_qty"]), 2))
print("WMAPE:", round(wmape(daily_result[TARGET_COL], daily_result["predicted_order_qty"]), 2))

print("\n====================")
print("HOURLY METRICS")
print("====================")
print("MAPE :", round(mape(hourly_result[TARGET_COL], hourly_result["predicted_order_qty"]), 2))
print("WMAPE:", round(wmape(hourly_result[TARGET_COL], hourly_result["predicted_order_qty"]), 2))

# ---------------------------------------------------------
# 11) SAVE OUTPUTS
# ---------------------------------------------------------
daily_result.to_csv("daily_forecast_result.csv", index=False)
hourly_result.to_csv("hourly_forecast_result.csv", index=False)

print("\nSaved:")
print("- daily_forecast_result.csv")
print("- hourly_forecast_result.csv")